# Regional and Route Analysis

## Goal

1. How are facilities and routes distributed geographically?
2. Does delivery performance (on-time, detention) differ by facility?
3. Do individual facilities stand out, or is the spread consistent with chance?
4. Which lanes/routes are the most and least profitable?
5. How does revenue relate to route distance and rate structure?

## Data Basis

**Via pipeline** (`clean_regional_analysis.csv`, one row per load):
`loads`, `routes`, `trips` and the pickup/delivery events (pivoted into `pickup_*` / 
`delivery_*` columns), plus the delivery metrics and an estimated fuel cost per load
(`fuel_cost`, aggregated per trip from `fuel_purchases`).

**Loaded directly in this notebook:**
- `facilities`: facility locations and types
- `delivery_events`: raw events, one row per event, restricted to the loads kept by the pipeline (for the facility analysis)
- `routes`, `fuel_purchases`, `trips`: raw tables for the data quality checks
- `maintenance_records`: for the cost estimate in section 6 (not loaded yet)

## 1. Load Data (via Pipeline)

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
sns.set_theme(style="darkgrid")

from delivery_pipeline import run_pipeline, PROCESSED_DATA_PATH_REGIONAL, RAW_DATA_DIR

logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

clean_df = run_pipeline(
    output_path=PROCESSED_DATA_PATH_REGIONAL,
    include_routes=True,
    include_fleet=False,
    include_costs=True
    )

print(f"Loads used for analysis: {len(clean_df)}")

fuel_df = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
facility_df = pd.read_csv(RAW_DATA_DIR / "facilities.csv")
routes_df = pd.read_csv(RAW_DATA_DIR / "routes.csv")
trips_df = pd.read_csv(RAW_DATA_DIR / "trips.csv")
delivery_df = pd.read_csv(RAW_DATA_DIR / "delivery_events.csv")


### 1.1 Verification of the fuel cost merge

In [ ]:
display(fuel_df["trip_id"].value_counts().describe())
display((fuel_df["total_cost"]<=0).value_counts())
display(fuel_df["trip_id"].isna().sum())
display(fuel_df["total_cost"].isna().sum())
display(trips_df["trip_id"].isin(fuel_df["trip_id"]).mean())
display(clean_df[["fuel_cost", "fuel_gallons_purchased", "n_fuel_purchases"]].head())

In [ ]:
fuel_sum = fuel_df["total_cost"].sum().round(2)
fuel_sum_merge = clean_df["fuel_cost"].sum().round(2)
diff = abs(fuel_sum-fuel_sum_merge).round(2)
removed = fuel_df.loc[~fuel_df["trip_id"].isin(clean_df["trip_id"]), "total_cost"].sum().round(2)
price = (clean_df["fuel_cost"].sum()/clean_df["fuel_gallons_purchased"].sum()).round(3)
avg_price = fuel_df["price_per_gallon"].mean().round(3)
has_fuel = clean_df["fuel_cost"].notna()

print(f"Sum of fuel_df: {fuel_sum}\nSum of clean_df: {fuel_sum_merge}\nDifference: {diff}\nRemoved values: {removed}")
print(f"Price per gallon (calculated): {price}, Price per gallon (fuel_df): {avg_price}")
print(clean_df.loc[has_fuel, ["fuel_gallons_purchased", "fuel_gallons_used"]].sum())

*The fuel cost merge is consistent: 90.1 % of loads have fuel data, and the difference between the raw and merged cost sums (574,953.83) equals exactly the fuel cost of the loads removed by the pipeline. The calculated price per gallon (3.899) matches the raw average (3.898). On loads with fuel data, about 43 % more gallons were purchased (24.4 M) than used (17.1 M).*

## 2. Data Quality Checks
### 2.1 Do all facility_id / route_id references resolve?

In [ ]:
unique_fac = facility_df["facility_id"].is_unique
unique_routes = routes_df["route_id"].is_unique
print(f"facility_id values are unique: {unique_fac}\nroute_id values are unique: {unique_routes}")

In [ ]:
def check_reference(relation, child, column, parent, parent_key):
    """Count rows, missing keys and orphans for one foreign key."""
    keys = child[column].dropna()
    return {
        "relation": relation,
        "rows": len(child),
        "missing_keys": child[column].isna().sum(),
        "orphans": (~keys.isin(parent[parent_key])).sum(),
    }

results = pd.DataFrame([
    check_reference("loads.route_id -> routes", clean_df, "route_id", routes_df, "route_id"),
    check_reference("events.facility_id -> facilities", delivery_df, "facility_id", facility_df, "facility_id"),
    check_reference("loads.pickup_facility_id -> facilities", clean_df, "pickup_facility_id", facility_df, "facility_id"),
    check_reference("loads.delivery_facility_id -> facilities", clean_df, "delivery_facility_id", facility_df, "facility_id"),
])
results["orphan_share"] = results["orphans"] / results["rows"]
display(results)

*All `route_id` references (84,924 loads), `facility_id` references (170,820 events) and the `pickup_facility_id` / `delivery_facility_id` columns of the loads (84,924 each) resolve; there are no orphans and no missing keys.*

### 2.2 Are location_city / location_state pairs plausible?

In [ ]:
sources = {
    "delivery_events": (delivery_df, "location_city", "location_state"),
    "fuel_purchases": (fuel_df, "location_city", "location_state"),
    "facilities": (facility_df, "city", "state"),
}
rows = []
for name, (df, city_col, state_col) in sources.items():
    rows.append({
        "table": name,
        "pairs": len(df[[city_col, state_col]].drop_duplicates()),
        "cities": df[city_col].nunique(),
        "max_states_per_city": df.groupby(city_col)[state_col].nunique().max(),
    })
display(pd.DataFrame(rows))

In [ ]:
merged = delivery_df.merge(facility_df[["facility_id", "city", "state"]], on="facility_id", how="left")
ok = (merged["location_city"] == merged["city"]) & (merged["location_state"] == merged["state"])
print(f"Events matching their facility: {ok.mean():.2%} ({ok.sum()} of {len(ok)})")

# every facility has events in all 20 cities (min = max = 20)
delivery_df.groupby("facility_id")["location_city"].nunique().describe()

In [ ]:
# Chance baseline: how often would an event's city equal its facility's city
# if event cities were assigned independently of the facility?
# Compared with the observed match rate above (city and state): every city match
# is also a state match here, so the city-only rate is the same (3.45 %).

p_event = delivery_df["location_city"].value_counts(normalize=True)
p_fac = merged["city"].value_counts(normalize=True)
print(f"Match expected by chance (city): {(p_event * p_fac).sum():.2%}")
print(f"Observed city-only match: {(merged['location_city'] == merged['city']).mean():.2%}")

In [ ]:
loc_vs_route = pd.Series({
    "pickup city == origin_city": (clean_df["pickup_location_city"] == clean_df["origin_city"]).mean(),
    "delivery city == destination_city": (clean_df["delivery_location_city"] == clean_df["destination_city"]).mean(),
    "pickup state == origin_state": (clean_df["pickup_location_state"] == clean_df["origin_state"]).mean(),
    "delivery state == destination_state": (clean_df["delivery_location_state"] == clean_df["destination_state"]).mean(),
})
display(loc_vs_route)

### Data Quality Note

* `delivery_events`: the location fields agree with the route in 100 % of loads (pickup city = origin_city, delivery city = destination_city), but only 3.45 % of events match the city/state of their assigned facility (about what chance gives, 3.50 %). Every facility has events in all 20 cities, so the facility assignment is geographically unrelated to where the event took place.
* `fuel_purchases`: 575 city/state pairs, every city paired with 23 states, so the fields are effectively random (confirms notebook 3).
* `facility_name` is not unique (35 distinct names for 50 facilities) and its type word does not match `facility_type`; `facility_id` and `facility_type` are used
* **Decision**: regions are derived from the routes (origin/destination). Facilities are analysed as units by facility_id; their home location is not interpreted as the location of the event.

## 3. Facility and Route Overview
### 3.1 Where are the facilities located, and what do routes look like?

In [ ]:
fac_per_state = facility_df.groupby("state").size().sort_values().reset_index(name="count")
fac_per_city = facility_df.groupby("city").size().sort_values().reset_index(name="count")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
        data=fac_per_state,
        x="state",
        y="count",
        ax=axes[0]
)
axes[0].set_xlabel("State")
axes[0].set_ylabel("Number of Facilities")
axes[0].set_title("Facilities per State")

sns.barplot(
        data=fac_per_city,
        x="city",
        y="count",
        ax=axes[1]
)
axes[1].set_xlabel("City")
axes[1].set_ylabel("Number of Facilities")
axes[1].set_title("Facilities per City")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Overview Facilities per State/City")
plt.tight_layout()
plt.show()



In [ ]:
fac_type = facility_df["facility_type"].value_counts()
print(fac_type)
fac_in_state = pd.crosstab(facility_df["state"], facility_df["facility_type"])
fac_in_city = pd.crosstab(facility_df["city"], facility_df["facility_type"])
display(fac_in_state)
display(fac_in_city)

In [ ]:
mileage = routes_df["typical_distance_miles"]
mileage.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(mileage, bins=15, kde=True, ax=ax)
ax.axvline(
    mileage.median(),
    color="red", 
    linestyle="--",
    label=f"Median {mileage.median():.1f}"
)
ax.set_xlabel("Miles")
ax.set_ylabel("Number of Routes")
ax.set_title("Distribution of Typical Distance")
ax.legend()
plt.show()

In [ ]:
print(f"Routes: {len(routes_df)}, distinct origin-destination pairs: {len(routes_df[['origin_city', 'destination_city']].drop_duplicates())}")

*The 50 facilities are spread over 20 states and 21 cities, with at most five per state (TX, TN) and per city (Nashville). The 58 routes range from 92 to 3,141 miles (median 1,271).
Every origin-destination pair occurs exactly once (58 routes = 58 lanes), so there are no repeated lanes to compare; this structure is most likely an artifact of the simulation.
Cross-docks are the most common facility type (20 of 50), followed by distribution centers (13), terminals (10) and warehouses (7).*

## 4. Delivery Performance by Facility
### Event-level table (for 4.1 and 4.3)
*Built in the notebook because only this notebook needs one row per event; the pipeline stays at one row per load.*


In [ ]:
events = delivery_df[delivery_df["load_id"].isin(clean_df["load_id"])].copy()
events = events.merge(
    facility_df[["facility_id", "facility_name", "facility_type", "city", "state"]]
        .rename(columns={"city": "facility_city", "state": "facility_state"}),
    on="facility_id", how="left", validate="many_to_one",
)
# location fields only duplicate the route's origin/destination (see 2.2)
events = events.drop(columns=["location_city", "location_state"])

print(len(events), 2 * len(clean_df))    # both 169,848 (two events per load)
print(events["facility_city"].isna().sum())       # 0

### 4.1 Does on-time rate / detention differ by facility?

In [ ]:
# on_time_flag is bool; detention_minutes has no missing values (checked)
facility_stats = (
    events
    .groupby("facility_id")
    .agg(
        n_events=("event_id", "count"),
        on_time_rate=("on_time_flag", "mean"),
        mean_detention=("detention_minutes", "mean")
        )
    .reset_index()
)

facility_stats = facility_stats.merge(
    facility_df[["facility_id","facility_name", "facility_type", "city"]],
    on="facility_id", how="left", validate="many_to_one",
)

facility_stats["label"] = facility_stats["city"] + " " + facility_stats["facility_id"].str[-2:]
overall_otr = events["on_time_flag"].mean()
overall_det = events["detention_minutes"].mean()
facility_stats["se"] = (
    np.sqrt(facility_stats["on_time_rate"] * (1 - facility_stats["on_time_rate"]) / facility_stats["n_events"])
    )

assert facility_stats["n_events"].sum() == len(events)
display(facility_stats.describe().round(3))


In [ ]:
plot_df = facility_stats.sort_values("on_time_rate", ascending=False)

y = range(len(plot_df))
fig, ax = plt.subplots(figsize=(8, 12))

ax.errorbar(
    x=plot_df["on_time_rate"], y=y, xerr=1.96 * plot_df["se"],
    fmt="o", color="C0", ecolor="gray", capsize=2,
)
ax.axvline(
    overall_otr, color="red", linestyle="--", 
    label=f"Overall: {overall_otr:.1%}")

ax.set_yticks(list(y))
ax.set_yticklabels(plot_df["label"])
ax.invert_yaxis()          
ax.set_xlim(0.50, 0.62) 

ax.set_xlabel("Avg. On-Time Rate")
ax.set_ylabel("Facility")
ax.set_title("On-Time Rate per Facility")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
plot_df = facility_stats.sort_values("mean_detention", ascending=False)

y = range(len(plot_df))
fig, ax = plt.subplots(figsize=(8, 12))

ax.axvline(
    overall_det, color="red", linestyle="--", 
    label=f"Overall: {overall_det.round(2)} min"
    )
ax.plot(plot_df["mean_detention"], y, "o")
ax.set_yticks(list(y))
ax.set_yticklabels(plot_df["label"])
ax.invert_yaxis()
ax.set_xlim(88, 94)

ax.set_xlabel("Avg. Detention (min)")
ax.set_ylabel("Facility")
ax.set_title("Detention in Minutes per Facility")
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
fac_per_type = (
    facility_stats
    .groupby("facility_type")
    .agg(
        n_facilities=("facility_id", "count"),
        on_time_rate=("on_time_rate", "mean"),
        mean_detention=("mean_detention", "mean"),
    )
    .reset_index()
)
display(fac_per_type.round(3))

overall_otr = events["on_time_flag"].mean()
overall_det = events["detention_minutes"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.stripplot(
    data=facility_stats, 
    x="facility_type", 
    y="on_time_rate", 
    ax=axes[0]
    )

axes[0].axhline(
    overall_otr, 
    color="red", 
    linestyle="--", 
    label=f"Overall: {overall_otr:.1%}"
    )
axes[0].set_xlabel("Facility Type")
axes[0].set_ylabel("Avg. On-Time Rate")
axes[0].set_title("On-Time Rate per Facility Type")
axes[0].legend()

sns.stripplot(
    data=facility_stats,
    x="facility_type",
    y="mean_detention",
    ax=axes[1]
    )

axes[1].axhline(
    overall_det.round(2), 
    color="red", 
    linestyle="--", 
    label=f"Overall: {overall_det.round(2)} min"
    )
axes[1].set_xlabel("Facility Type")
axes[1].set_ylabel("Avg. Detention (Minutes)")
axes[1].set_title("Detention per Facility Type")
axes[1].legend()

plt.tight_layout()
plt.show()

*Interpretation*
- *On-time rate: The facility rates range from 53.2 % to 58.3 % (overall 55.7 %), a spread of about 5 percentage points. With roughly 3,400 events per facility the 95 % intervals are about ±1.7 pp wide, so most facilities cannot be told apart from the average. Only the extremes (e.g. Houston 01 vs. Indianapolis 48) have non-overlapping intervals, but extremes of 50 facilities also differ somewhat by chance.*
- *Detention: Mean detention ranges from 88.7 to 93.8 minutes (overall 91.5), i.e. about 3 % around the average.*
- *Facility type: Averages by type differ by less than one percentage point (on-time rate) and less than one minute (detention), which is small compared with the spread between individual facilities; there is no clear type effect.*
- *Whether the spread between facilities is larger than expected by chance is tested in 4.2.*

### 4.2 Is that spread larger than expected by chance?

**Method (permutation test)**

*Question:* Is the spread of on-time rates between facilities larger than expected if the facility had no influence?

1. Compute the observed spread: the standard deviation of the 50 facility on-time rates.
2. Shuffle `on_time_flag` randomly across all events. This breaks any link between facility and outcome, while the number of events per facility stays the same.
3. Recompute the on-time rate per facility and the standard deviation of these rates.
4. Repeat 5,000 times to get the distribution of the spread under "no facility effect".
5. The p-value is the share of shuffled spreads that are at least as large as the observed one (with a +1 correction, so p is never exactly 0).

The same procedure is applied to the mean detention per facility. Random seed: 42.

In [ ]:
def permutation_spread(values, codes, n_iterations=5000, seed=42):
    """Observed spread (std of group means), null distribution and p-value."""
    counts = np.bincount(codes)
    observed = (np.bincount(codes, weights=values) / counts).std(ddof=1)
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_iterations):
        shuffled = rng.permutation(values)
        null.append((np.bincount(codes, weights=shuffled) / counts).std(ddof=1))
    null = np.array(null)
    p_value = (np.sum(null >= observed) + 1) / (n_iterations + 1)
    return observed, null, p_value

In [ ]:
codes, _ = pd.factorize(events["facility_id"])
flags = events["on_time_flag"].to_numpy(dtype=float)
minutes = events["detention_minutes"].to_numpy(dtype=float)

obs_otr, null_otr, p_otr = permutation_spread(flags, codes)
obs_det, null_det, p_det = permutation_spread(minutes, codes)

In [ ]:
results = pd.DataFrame({
    "metric": ["On-Time Rate", "Detention (min)"],
    "observed_std": [obs_otr, obs_det],
    "null_mean_std": [null_otr.mean(), null_det.mean()],
    "p_value": [p_otr, p_det],
})
display(results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
panels = [
    (axes[0], null_otr, obs_otr, p_otr, "On-Time Rate"),
    (axes[1], null_det, obs_det, p_det, "Detention"),
]
for ax, null, obs, p, name in panels:
    sns.histplot(null, bins=40, ax=ax)
    ax.axvline(obs, color="red", linestyle="--", label=f"Observed: {obs:.4f}")
    ax.set_title(f"{name}: Spread Between Facilities (p = {p:.4f})")
    ax.set_xlabel("Std of Facility Means Under Shuffling")
    ax.legend()
plt.tight_layout()
plt.show()

*Interpretation*
- *On-time rate: The spread between facilities (standard deviation 1.15 pp) is clearly larger than what random shuffling produces (about 0.85 pp on average). Almost none of the 5,000 shuffled runs reached the observed value (p = 0.0004), so the differences between facilities are unlikely to be pure chance.*
- *Size: The effect is small. A typical facility deviates by about 1 pp from the overall rate of 55.7 %, and even the best and worst facility are only about 5 pp apart.*
- *Detention: The observed spread (1.13 min) lies in the middle of the shuffled distribution (mean 1.17 min, p = 0.63). There is no evidence that facilities differ in detention beyond chance.*
- *Caveats: The test shows that facilities differ in on-time rate, not which ones; the chart in 4.1 only shows that the extremes lie apart. Both events of a load enter the test as if they were independent, which can make the p-value look somewhat too small. Since the data are simulated, the result reflects how they were generated rather than a real operation.*

### 4.3 Does on-time rate differ by event type (pickup vs. delivery)?

In [ ]:
event_type_summary = (
    events.groupby("event_type")
    .agg(
        on_time_rate=("on_time_flag", "mean"),
        avg_det_mins=("detention_minutes", "mean")
    )
    .reset_index()
    .assign(
        on_time_rate=lambda df: (df["on_time_rate"] * 100).round(2),
        avg_det_mins=lambda df: df["avg_det_mins"].round(2)
    )
)

display(event_type_summary)
by_type = event_type_summary.set_index("event_type")
diff = by_type.loc["Pickup"] - by_type.loc["Delivery"]
rel = (by_type.loc["Pickup"] / by_type.loc["Delivery"] - 1) * 100

print(f"On-time rate, Pickup - Delivery: {diff['on_time_rate']:+.1f} pp ({rel['on_time_rate']:+.1f} % relative)")
print(f"Detention, Pickup - Delivery: {diff['avg_det_mins']:+.1f} min ({rel['avg_det_mins']:+.1f} % relative)")


In [ ]:
g = events.groupby("event_type")
n = g.size()

p = g["on_time_flag"].mean()
se = np.sqrt((p * (1 - p) / n).sum())
print(f"On-time rate, Pickup - Delivery: {(p['Pickup'] - p['Delivery']) * 100:+.1f} pp +/- {1.96 * se * 100:.1f} pp (95 % interval)")

m, s = g["detention_minutes"].mean(), g["detention_minutes"].std()
se_d = np.sqrt((s ** 2 / n).sum())
print(f"Detention, Pickup - Delivery: {m['Pickup'] - m['Delivery']:+.1f} min +/- {1.96 * se_d:.1f} min (95 % interval)")

In [ ]:
late_min = (pd.to_datetime(events["actual_datetime"]) - pd.to_datetime(events["scheduled_datetime"])).dt.total_seconds() / 60
print(late_min[events["on_time_flag"]].abs().max())    # about 120
print(late_min[~events["on_time_flag"]].abs().min())   # about 120
print(late_min.groupby(events["event_type"]).agg(["min", "max"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6))

sns.barplot(
    data=event_type_summary,
    x="event_type",
    y="on_time_rate",
    ax=axes[0]
)
axes[0].set_xlabel("Event Type")
axes[0].set_ylabel("On-Time Rate (%)")
axes[0].set_title("On-Time Rate (%) per Event Type")

sns.barplot(
    data=event_type_summary,
    x="event_type",
    y="avg_det_mins",
    ax=axes[1]
)
axes[1].set_xlabel("Event Type")
axes[1].set_ylabel("Avg. Detention (min)")
axes[1].set_title("Avg. Detention (min) per Event Type")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (name, group) in zip(axes, events.groupby("event_type")):
    mean = group["detention_minutes"].mean()
    sns.histplot(group["detention_minutes"], binwidth=10, ax=ax)
    ax.axvline(mean, color="red", linestyle="--", label=f"Mean: {mean:.1f}")
    ax.set_title(f"Detention per {name} Event")
    ax.set_xlabel("Detention (min)")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.ecdfplot(data=events, x="detention_minutes", hue="event_type", ax=ax)
ax.set_xlabel("Detention (min)")
ax.set_ylabel("Share of Events up to This Value")
ax.set_title("Cumulative Distribution of Detention per Event Type")
plt.tight_layout()
plt.show()

In [ ]:
pickup_share = events.groupby("facility_id")["event_type"].apply(lambda s: (s == "Pickup").mean())
print(pickup_share.describe().round(3))

*Interpretation*
- *On-time rate: Pickups are on time in 66.7 % of events, deliveries in 44.7 %, a difference of 22.0 pp (95 % interval about ±0.5 pp). With about 85,000 events per type this is far from chance.*
- *Definition: `on_time_flag` is true if the actual time deviates by at most 2 hours from the schedule; it does not mean "not late". This is why the rate differs from the share of delayed deliveries in notebook 1, where any delay counts.*
- *Likely reason: In the data, deviations from the schedule are spread over a wider range for deliveries (about −3 h to +6 h) than for pickups (−2 h to +4 h), so more deliveries fall outside the 2-hour window.*
- *Detention: Deliveries wait 30.1 min longer on average (106.6 vs. 76.5 min). The cumulative curves are almost straight lines, i.e. detention is spread evenly up to an upper limit of about 180 min for pickups and 240 min for deliveries; about 15 % of pickups and 11 % of deliveries have no detention at all. The difference in the mean follows directly from the different upper limits.*
- *Consequence for 4.1/4.2: The facility comparison pools both event types, which differ strongly. A different mix of pickups and deliveries per facility could therefore distort the comparison. The share of pickups per facility is close to 50 % (range 48–53 %), which shifts a facility's on-time rate by about 0.5 pp at most, much less than the observed spread of about 1.2 pp.*
- *Caveats: Pickup and delivery of one load are not independent events. Since the data are simulated, the difference reflects how they were generated.*

## 5. Regional Delivery Performance
### 5.1 Does delay/duration differ by region (state)?

*Region = destination state (as in notebook 1); the origin state is shown for comparison.*

In [ ]:
# event location corresponds to the destination state or origin state of the route
display((clean_df["destination_state"] == clean_df["delivery_location_state"]).value_counts())
display((clean_df["origin_state"] == clean_df["pickup_location_state"]).value_counts())

delay_duration_by_destination = (
    clean_df.groupby("destination_state")
    .agg(
        n_loads=("load_id", "count"),
        delay_rate=("is_delayed_delivery", "mean"),
        avg_duration=("delivery_duration_hours", "mean")
    )
    .reset_index()
    .sort_values("avg_duration", ascending=False)
)

delay_duration_by_origin = (
    clean_df.groupby("origin_state")
    .agg(
        n_loads=("load_id", "count"),
        delay_rate=("is_delayed_delivery", "mean"),
        avg_duration=("delivery_duration_hours", "mean")
    )
    .reset_index()
    .sort_values("avg_duration", ascending=False)
)

display(delay_duration_by_destination.round(2))
display(delay_duration_by_origin.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, col in zip(axes, ["origin_state", "destination_state"]):
    stats = clean_df.groupby(col)["delivery_duration_hours"].mean().sort_values(ascending=False)
    sns.barplot(x=stats.index, y=stats.values, ax=ax)
    ax.set_title(f"Avg. Delivery Duration per {col.replace('_', ' ').title()}")
    ax.set_xlabel("State")
axes[0].set_ylabel("Avg. Delivery Duration (h)")
plt.tight_layout()
plt.show()

*The delivery duration per destination state ranges from 11.9 h (TN) up to 40.4 h (WA and OR), which is a factor of 3.4. Per origin state the range is from 7.8 h (NY) to 43.1 h (WA), which is a factor of 5.5. The delay rate however ranges from 66 % to 71 % per destination state and from 65 % to 72 % per origin state, with NY the highest at both.
Caveat: Distances are not taken into consideration, this is examined in 5.2*

### 5.2 Does the regional duration difference remain after accounting for distance?

In [ ]:
cols = ["delivery_duration_hours", "actual_duration_hours"]
display(clean_df[cols].describe().round(2))
display(clean_df[cols].corr().round(3))
display((clean_df[cols[0]] - clean_df[cols[1]]).abs().describe().round(2))

In [ ]:
n_zero = (clean_df["delivery_duration_hours"] <= 0).sum()
print(f"Loads with zero duration: {n_zero} ({n_zero / len(clean_df):.2%})")

dist_df = clean_df[clean_df["delivery_duration_hours"] > 0].copy()
dist_df["hours_per_100_miles"] = dist_df["delivery_duration_hours"] / dist_df["actual_distance_miles"] * 100
dist_df["avg_speed_mph"] = dist_df["actual_distance_miles"] / dist_df["delivery_duration_hours"]
display(dist_df[["hours_per_100_miles", "avg_speed_mph"]].describe().round(2))
display(dist_df["avg_speed_mph"].quantile([0.001, 0.01, 0.99, 0.999]).round(1))

display((clean_df["actual_distance_miles"] / clean_df["typical_distance_miles"] - 1).describe())
display(dist_df.nsmallest(10, "delivery_duration_hours")[["actual_distance_miles", "delivery_duration_hours", "avg_speed_mph"]])
for limit in (80, 100, 150):
    print(f"Loads with speed > {limit} mph:", (dist_df["avg_speed_mph"] > limit).sum())

fast = dist_df["avg_speed_mph"] > 100
display(dist_df.loc[fast, "actual_distance_miles"].describe().round(0))   # are they all short routes?

by_state = pd.DataFrame({
    "all": dist_df.groupby("destination_state")["hours_per_100_miles"].mean(),
    "without_fast": dist_df[~fast].groupby("destination_state")["hours_per_100_miles"].mean(),
})
by_state["diff"] = by_state["all"] - by_state["without_fast"]
display(by_state.round(3).sort_values("diff"))


*Data quality (5.2): Three loads with zero duration are excluded, since speed is undefined for them. A further 705 loads (0.83 %) have an implausible speed above 100 mph; their pickup and delivery timestamps are almost identical, and they occur on short routes (90–546 miles, median 166). They are kept: excluding them raises the state means of hours per 100 miles for only four states (NY +0.08, WA +0.05, PA +0.05, CA +0.02, at most about 3 %). Actual and typical route distance differ by −2 % to +8 %, so the actual distance is used. `delivery_duration_hours` and the trip's `actual_duration_hours` are highly correlated (0.98) but differ by 2.5 h on average (up to 6 h); the delivery duration is used for comparability with 5.1.*

## 6. Route / Lane Profitability
### 6.1 What is the revenue per route?

In [ ]:
# TODO: aggregate loads.revenue by route_id

### 6.2 What are the approximate costs per route?

In [ ]:
# TODO: estimate fuel cost per route (via trips -> fuel_purchases, aggregated per trip then per route)
# TODO: optionally add a share of maintenance cost per mile (from notebook 3 / maintenance_records)
# TODO: check whether missing fuel data is random (compare trip distance with/without fuel entry)


### Data Quality Note

*TODO: note the assumptions/approximations made when allocating fuel and maintenance cost to routes.*

### 6.3 Which lanes are most and least profitable?

In [ ]:
# TODO: margin = revenue - estimated cost, per route; rank; chart
# TODO: compute margin per load, then average; do not fill missing fuel_cost with 0

*TODO: interpretation.*

## 7. Conclusion

*TODO: summarize facility-level findings, the pickup-vs-delivery difference, the distance-adjusted regional comparison, and route profitability; link to notebook 5.*